# Pipeline completo de ML con scikit-learn

Este notebook recorre el flujo de trabajo completo de un proyecto de Machine Learning supervisado usando scikit-learn: desde la exploración de datos hasta el despliegue de un pipeline reproducible.

**Flujo de trabajo:**
1. Cargar y explorar datos (EDA)
2. Preprocesamiento
3. Modelo baseline
4. Modelos más potentes
5. Validación cruzada
6. Optimización de hiperparámetros
7. Evaluación final
8. Pipeline reproducible

**Requisitos:**
```bash
pip install scikit-learn matplotlib seaborn joblib
```

## 1. Cargar y explorar datos

La **Exploratory Data Analysis (EDA)** es el primer paso en cualquier proyecto de ML. Antes de entrenar un modelo, necesitamos entender:

- **Dimensiones**: cuántas muestras y features tenemos
- **Tipos de datos**: numéricos, categóricos, binarios
- **Distribuciones**: cómo se distribuyen las features y el target
- **Valores faltantes**: cuántos y en qué columnas
- **Balance de clases**: crucial para clasificación
- **Correlaciones**: relaciones entre variables

Usaremos el dataset de **Breast Cancer** de scikit-learn (clasificación binaria: maligno vs benigno).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Load the dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print(f"Dataset shape: {X.shape}")
print(f"Features: {X.columns.tolist()[:10]}... ({len(X.columns)} total)")
print(f"\nTarget names: {data.target_names}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True).round(3).to_dict()}")

print(f"\n=== Statistical Summary (first 5 features) ===")
print(X.iloc[:, :5].describe().round(3))

print(f"\nMissing values: {X.isnull().sum().sum()}")

In [ ]:
# Visualize feature distributions and correlations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Feature distributions for top features
top_features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area']
for i, feat in enumerate(top_features):
    ax = axes[i // 2, i % 2]
    for label in [0, 1]:
        subset = X[y == label][feat]
        ax.hist(subset, bins=25, alpha=0.6, label=data.target_names[label], edgecolor='black')
    ax.set_title(f'Distribution: {feat}', fontsize=11)
    ax.legend()
    ax.set_xlabel(feat)

plt.suptitle('Distribución de features por clase', fontsize=14)
plt.tight_layout()
plt.show()

# 2. Correlation matrix of first 10 features
fig, ax = plt.subplots(figsize=(10, 8))
corr = X.iloc[:, :10].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación (primeras 10 features)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Preprocesamiento

El preprocesamiento transforma los datos crudos en un formato óptimo para los modelos de ML.

**Pasos comunes:**
- **Train/test split**: separar datos antes de cualquier transformación para evitar data leakage
- **Scaling**: normalizar features numéricas (StandardScaler, MinMaxScaler)
- **Encoding**: convertir variables categóricas a numéricas (OneHotEncoder, LabelEncoder)
- **Feature selection**: eliminar features redundantes o irrelevantes

> **Data Leakage**: nunca ajustar (fit) el scaler con datos de test. Fit en train, transform en ambos.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Stratified split to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test class balance:  {y_test.value_counts(normalize=True).round(3).to_dict()}")

# Scale features - fit ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)         # only transform!

print(f"\n--- Before scaling ---")
print(f"Train mean (first 3): {X_train.iloc[:, :3].mean().round(2).values}")
print(f"Train std  (first 3): {X_train.iloc[:, :3].std().round(2).values}")

print(f"\n--- After scaling ---")
print(f"Train mean (first 3): {X_train_scaled[:, :3].mean(axis=0).round(4)}")
print(f"Train std  (first 3): {X_train_scaled[:, :3].std(axis=0).round(4)}")

## 3. Modelo Baseline

Siempre empezar con un modelo simple como baseline. La **Regresión Logística** es ideal porque:
- Es rápida de entrenar
- Fácil de interpretar (coeficientes = importancia de features)
- Funciona bien con features escaladas
- Proporciona probabilidades calibradas

El baseline establece el mínimo de rendimiento que modelos más complejos deben superar.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Train baseline model
baseline = LogisticRegression(max_iter=10000, random_state=42)
baseline.fit(X_train_scaled, y_train)

# Evaluate
y_pred_baseline = baseline.predict(X_test_scaled)
y_prob_baseline = baseline.predict_proba(X_test_scaled)[:, 1]

print("=== Baseline: Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=data.target_names))

# Top important features
feature_importance = pd.Series(
    np.abs(baseline.coef_[0]), index=data.feature_names
).sort_values(ascending=False)

print("\nTop 10 most important features (by |coefficient|):")
print(feature_importance.head(10).round(4))

## 4. Modelos más potentes

Una vez tenemos un baseline, probamos modelos más complejos. **Random Forest** es un excelente siguiente paso porque:
- No requiere scaling de features
- Maneja bien relaciones no lineales
- Proporciona feature importance
- Es robusto a outliers
- Es un ensemble (combina múltiples árboles de decisión)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

# Evaluate
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=data.target_names))

# Compare with baseline
print("\n=== Comparison ===")
print(f"Baseline (LogReg): {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Random Forest:     {accuracy_score(y_test, y_pred_rf):.4f}")

## 5. Cross-Validation

Una sola división train/test puede dar resultados engañosos. La **validación cruzada** divide los datos en K folds y entrena K veces, usando cada fold como validación una vez.

**Ventajas:**
- Estimación más robusta del rendimiento real
- Varianza del rendimiento (¿el modelo es estable?)
- Usa todos los datos para entrenamiento y validación

Usamos **StratifiedKFold** para mantener el balance de clases en cada fold.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

# Define cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compare multiple models
models = {
    'LogisticRegression': LogisticRegression(max_iter=10000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

print("=== 5-Fold Cross-Validation Results ===")
print(f"{'Model':<25} {'Mean Accuracy':>15} {'Std':>10}")
print("-" * 52)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<25} {scores.mean():>15.4f} {scores.std():>10.4f}")

# Visualize CV results
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(cv_results.values(), labels=cv_results.keys())
ax.set_title('Comparación de modelos (5-Fold CV)', fontsize=13)
ax.set_ylabel('Accuracy')
ax.set_ylim(0.9, 1.0)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning

Los hiperparámetros controlan el comportamiento del algoritmo de aprendizaje (no se aprenden de los datos). Buscar la combinación óptima puede mejorar significativamente el rendimiento.

**Métodos:**
- **GridSearchCV**: prueba todas las combinaciones (exhaustivo, lento)
- **RandomizedSearchCV**: prueba combinaciones aleatorias (más rápido, bueno para muchos parámetros)

> Siempre hacer tuning con cross-validation para evitar sobreajustar al validation set.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define hyperparameter search space
param_distributions = {
    'n_estimators': randint(50, 300),
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

# Randomized search with cross-validation
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=50,          # Number of random combinations to try
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train_scaled, y_train)

print("=== RandomizedSearchCV Results ===")
print(f"Best accuracy (CV): {random_search.best_score_:.4f}")
print(f"\nBest parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate best model on test set
best_rf = random_search.best_estimator_
y_pred_best = best_rf.predict(X_test_scaled)
print(f"\nTest accuracy (best model): {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Test accuracy (default RF): {accuracy_score(y_test, y_pred_rf):.4f}")

## 7. Evaluación final

La evaluación final debe ir más allá del accuracy. Para clasificación, necesitamos:

- **Confusion Matrix**: visualiza verdaderos/falsos positivos y negativos
- **ROC Curve**: rendimiento en todos los umbrales de clasificación (AUC = área bajo la curva)
- **Precision-Recall Curve**: especialmente útil con clases desbalanceadas

En diagnóstico médico (nuestro caso), un falso negativo (no detectar un tumor maligno) es mucho peor que un falso positivo.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# Get predictions from best model
y_pred_final = best_rf.predict(X_test_scaled)
y_prob_final = best_rf.predict_proba(X_test_scaled)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(cm, display_labels=data.target_names)
disp.plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusión', fontsize=13)

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob_final)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC', fontsize=13)
axes[1].legend(loc='lower right')

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob_final)
ap = average_precision_score(y_test, y_prob_final)
axes[2].plot(recall, precision, color='green', lw=2, label=f'PR (AP = {ap:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Curva Precision-Recall', fontsize=13)
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.show()

print(f"\n=== Final Model Report ===")
print(classification_report(y_test, y_pred_final, target_names=data.target_names))

## 8. Pipeline completo

Un **Pipeline** de scikit-learn encadena preprocesamiento + modelo en un solo objeto. Esto garantiza:

- **Reproducibilidad**: el mismo pipeline para train y test
- **No data leakage**: el scaler se ajusta solo en train durante cross-validation
- **Despliegue simple**: guardar y cargar un solo objeto

Es la forma correcta de poner un modelo en producción.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib

# Build the complete pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        **random_search.best_params_,  # Use best params from tuning
        random_state=42
    ))
])

# Cross-validate the full pipeline (no data leakage!)
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f"Pipeline CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Train on all training data
pipeline.fit(X_train, y_train)

# Predict on test data (scaler automatically applied)
y_pred_pipeline = pipeline.predict(X_test)
print(f"Pipeline Test Accuracy: {accuracy_score(y_test, y_pred_pipeline):.4f}")

# Save the pipeline
joblib.dump(pipeline, 'breast_cancer_pipeline.joblib')
print("\nPipeline saved to 'breast_cancer_pipeline.joblib'")

# Load and verify
loaded_pipeline = joblib.load('breast_cancer_pipeline.joblib')
y_pred_loaded = loaded_pipeline.predict(X_test)
print(f"Loaded pipeline accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")
print(f"Predictions match: {np.all(y_pred_pipeline == y_pred_loaded)}")

# Show pipeline structure
print(f"\n=== Pipeline Structure ===")
print(pipeline)

## Resumen y siguientes pasos

### Lo que hemos aprendido:
1. **EDA** es imprescindible antes de modelar
2. **Preprocesamiento** correcto evita data leakage
3. **Baseline** simple primero, luego modelos más complejos
4. **Cross-validation** para estimaciones robustas
5. **Hyperparameter tuning** para optimizar el rendimiento
6. **Evaluación** con múltiples métricas según el problema
7. **Pipeline** para reproducibilidad y despliegue

### Siguientes pasos:
- **Feature engineering**: crear features más informativas
- **Feature selection**: eliminar features redundantes (SelectKBest, Recursive Feature Elimination)
- **Ensemble methods**: stacking, blending
- **Interpretabilidad**: SHAP values, LIME
- **MLOps**: MLflow para tracking de experimentos, Docker para despliegue